# ETL — Cartão de Pagamento do Governo Federal (CPGF)

Neste notebook vamos integrar os dados de **2023, 2024 e 2025**, tratar os campos e montar um Data Warehouse em esquema estrela no SQLite.

Seguiremos três etapas: **extração, transformação e carregamento**. Cada linha dos CSVs será mantida como uma linha na tabela fato, inclusive repetições exatas.

Os arquivos já devem ter sido baixados pelo script `scripts/extract_cpgf.py`. Aqui, a extração consiste na leitura desses arquivos. Execute as células na ordem, preferencialmente com **Reiniciar kernel e executar tudo**.

## 1. Preparando o ambiente

Precisamos do pandas e das bibliotecas padrão do Python. Se necessário, execute `%pip install pandas` em uma célula antes de continuar.

Abra o notebook dentro do repositório. No Colab, clone o projeto e configure `RAIZ_CONFIGURADA`. Para uma demonstração menor, use `MODO_AMOSTRA = True`: o resultado será salvo em um banco separado.

In [1]:
from pathlib import Path
from decimal import Decimal
import hashlib
import json
import re
import sqlite3
import tempfile
import time

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print  # Permite validar as células fora do Jupyter.

RAIZ_CONFIGURADA = None  # Exemplo no Colab: Path('/content/projeto-integracao-dw')
MODO_AMOSTRA = False
ANOS = (2023, 2024, 2025)


def localizar_raiz(configurada=None):
    candidatos = [Path(configurada)] if configurada is not None else [Path.cwd(), *Path.cwd().parents]
    for pasta in candidatos:
        if (pasta / 'scripts/sql/ddl/schema.sql').is_file():
            return pasta.resolve()
    raise FileNotFoundError('Configure RAIZ_CONFIGURADA com a pasta do repositório.')


RAIZ = localizar_raiz(RAIZ_CONFIGURADA)
ARQUIVOS = [
    RAIZ / 'data/raw' / (f'amostras/{ano}_amostra.csv' if MODO_AMOSTRA else f'{ano}_CPGF.csv')
    for ano in ANOS
]
DESTINO = RAIZ / 'data/database' / ('cpgf_dw_amostra.sqlite' if MODO_AMOSTRA else 'cpgf_dw.sqlite')
DDL = RAIZ / 'scripts/sql/ddl/schema.sql'
INICIO = time.perf_counter()
print('pandas:', pd.__version__)
print('Modo:', 'amostra' if MODO_AMOSTRA else 'completo', '| Destino:', DESTINO)

pandas: 3.1.1
Modo: completo | Destino: /home/fagpaf/Projetos/projeto-integracao-dw/data/database/cpgf_dw.sqlite


## 2. Extraindo os dados

Vamos ler os arquivos separados por `;`. Todos os campos serão lidos inicialmente como texto para preservar zeros à esquerda e documentos mascarados.

O dicionário abaixo define nomes mais simples para as colunas. A função `exigir` interrompe a execução quando uma conferência falha, evitando continuar com dados inconsistentes.

In [2]:
COLUNAS = {
    'CÓDIGO ÓRGÃO SUPERIOR': 'codigo_orgao_superior',
    'NOME ÓRGÃO SUPERIOR': 'nome_orgao_superior',
    'CÓDIGO ÓRGÃO': 'codigo_orgao',
    'NOME ÓRGÃO': 'nome_orgao',
    'CÓDIGO UNIDADE GESTORA': 'codigo_unidade_gestora',
    'NOME UNIDADE GESTORA': 'nome_unidade_gestora',
    'ANO EXTRATO': 'ano_extrato',
    'MÊS EXTRATO': 'mes_extrato',
    'CPF PORTADOR': 'cpf_mascarado',
    'NOME PORTADOR': 'nome_portador',
    'CNPJ OU CPF FAVORECIDO': 'documento',
    'NOME FAVORECIDO': 'nome_favorecido',
    'TRANSAÇÃO': 'descricao_original',
    'DATA TRANSAÇÃO': 'data_original',
    'VALOR TRANSAÇÃO': 'valor_original',
}


def exigir(condicao, mensagem):
    if not bool(condicao):
        raise ValueError(mensagem)

### Lendo e juntando os três anos

Para cada arquivo, vamos conferir as colunas e o ano, remover espaços nas extremidades e registrar sua origem. Depois juntaremos os DataFrames em uma única tabela.

Também montaremos um manifesto com quantidade de registros, tamanho e hash de cada arquivo. O hash ajuda a identificar se a entrada mudou entre execuções.

In [3]:
lista_dataframes = []
registros_manifesto = []

for arquivo in ARQUIVOS:
    if not arquivo.is_file():
        raise FileNotFoundError(f'Arquivo não encontrado: {arquivo}')

    df_ano = pd.read_csv(arquivo, sep=';', encoding='utf-8-sig', dtype=str, keep_default_na=False)
    exigir(list(df_ano.columns) == list(COLUNAS), f'{arquivo.name}: cabeçalho diferente do esperado.')
    exigir(len(df_ano) > 0, f'{arquivo.name}: arquivo vazio.')
    df_ano = df_ano.rename(columns=COLUNAS)
    df_ano = df_ano.apply(lambda coluna: coluna.str.strip())
    df_ano['arquivo_origem'] = arquivo.name
    df_ano['registro_origem'] = range(1, len(df_ano) + 1)
    exigir(df_ano['ano_extrato'].eq(arquivo.name[:4]).all(), f'{arquivo.name}: ano divergente.')

    lista_dataframes.append(df_ano)
    registros_manifesto.append({
        'arquivo': arquivo.name, 'registros': len(df_ano),
        'bytes': arquivo.stat().st_size,
        'sha256': hashlib.sha256(arquivo.read_bytes()).hexdigest(),
    })

brutos = pd.concat(lista_dataframes, ignore_index=True)
manifesto = pd.DataFrame(registros_manifesto)
display(manifesto)
print('Total de registros:', len(brutos))

,arquivo,registros,bytes,sha256
0,2023_CPGF.csv,134122,37754611,b27b543e0f6c59d796bffc930e6f3d2426635bb486d840...
1,2024_CPGF.csv,143849,39887923,bd3fb71c14649923a3f161c4eb0a857aa155102f0288b9...
2,2025_CPGF.csv,160800,44962870,b689ba7d7135d28098811dbd7f8c0ee7c25429309f2ca7...


Total de registros: 438771


### Conferindo os metadados da extração

Na base completa, comparamos as quantidades de cada mês com o arquivo de metadados, quando disponível. As amostras não precisam conter todos os meses.

Diferenças de contagem geram um aviso e não interrompem a carga. Por decisão do projeto, todos os registros dos CSVs são preservados; a causa da divergência não foi investigada. As validações de períodos, integridade e valores antes e depois da carga continuam obrigatórias.

In [4]:
def conferir_metadados(dados, caminho):
    if not caminho.is_file():
        print('Metadados ausentes: contagens da extração não foram conferidas.')
        return
    metadados = json.loads(caminho.read_text(encoding='utf-8'))
    registros = [m for m in metadados if int(m['referencia'][:4]) in ANOS]
    esperado = {m['referencia']: m['qtd_registros'] for m in registros}
    periodos = {f'{ano}{mes:02d}' for ano in ANOS for mes in range(1, 13)}
    exigir(len(registros) == len(periodos) and set(esperado) == periodos,
           'Metadados devem conter uma entrada para cada mês solicitado.')
    obtido = (dados['ano_extrato'] + dados['mes_extrato'].str.zfill(2)).value_counts().to_dict()
    exigir(set(obtido) <= periodos, 'Há períodos fora do intervalo solicitado.')
    divergencias = [
        {'periodo': periodo, 'registros_csv': obtido.get(periodo, 0),
         'registros_metadados': quantidade}
        for periodo, quantidade in sorted(esperado.items())
        if obtido.get(periodo, 0) != quantidade
    ]
    if divergencias:
        print('AVISO: contagens mensais divergem dos metadados. '
              'A carga continuará preservando todos os registros dos CSVs.')
        display(pd.DataFrame(divergencias))
    else:
        print('Contagens mensais conferidas com os metadados.')

if not MODO_AMOSTRA:
    conferir_metadados(brutos, RAIZ / 'data/raw/metadados.json')

AVISO: contagens mensais divergem dos metadados. A carga continuará preservando todos os registros dos CSVs.


,periodo,registros_csv,registros_metadados
0,202505,12815,12814


## 3. Conhecendo os dados antes do tratamento

Vamos observar as primeiras linhas, os tipos das colunas e a quantidade de valores distintos.

In [5]:
display(brutos.head())
brutos.info()
display(brutos[list(COLUNAS.values())].nunique(dropna=False).rename('valores_distintos'))

,codigo_orgao_superior,nome_orgao_superior,codigo_orgao,nome_orgao,codigo_unidade_gestora,nome_unidade_gestora,ano_extrato,mes_extrato,cpf_mascarado,nome_portador,documento,nome_favorecido,descricao_original,data_original,valor_original,arquivo_origem,registro_origem
0,63000,Advocacia-Geral da União,63000,Advocacia-Geral da União - Unidades com víncul...,110161,SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO,2023,01,***.562.861-**,ANTONIO CARLOS MELO DOS SANTOS,24883608000170,IG PLOTTER GRAFICA RAPIDA LTDA,COMPRA A/V - R$ - APRES,29/11/2022,"162,48",2023_CPGF.csv,1
1,63000,Advocacia-Geral da União,63000,Advocacia-Geral da União - Unidades com víncul...,110161,SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO,2023,01,***.866.951-**,JONAS SCHOTTZ DA SILVA,04588580000130,OPCAO FERRAZ FERRAGISTA LTDA,COMPRA A/V - R$ - APRES,08/12/2022,"28,40",2023_CPGF.csv,2
2,63000,Advocacia-Geral da União,63000,Advocacia-Geral da União - Unidades com víncul...,110161,SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO,2023,01,***.866.951-**,JONAS SCHOTTZ DA SILVA,04588580000130,OPCAO FERRAZ FERRAGISTA LTDA,COMPRA A/V - R$ - APRES,07/12/2022,"133,10",2023_CPGF.csv,3
3,63000,Advocacia-Geral da União,63000,Advocacia-Geral da União - Unidades com víncul...,110161,SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO,2023,01,***.989.121-**,DANIELA RODRIGUES MACEDO FERREIRA,02828297000103,EXTINTORES ARAGUAIA LTDA,COMPRA A/V - R$ - APRES,05/12/2022,"855,00",2023_CPGF.csv,4
4,63000,Advocacia-Geral da União,63000,Advocacia-Geral da União - Unidades com víncul...,110161,SUPERINTENDENCIA REG. DE ADMIN. DA 1ª REGIAO,2023,01,***.337.052-**,DEMERVAL PANTOJA DA CRUZ,06596916000132,MACEDO COMERCIO DE PARAFUSOS E FERRAMENTAS LTDA,COMPRA A/V - R$ - APRES,30/11/2022,"35,00",2023_CPGF.csv,5


<class 'pandas.DataFrame'>
RangeIndex: 438771 entries, 0 to 438770
Data columns (total 17 columns):
 #   Column                  Non-Null Count   Dtype
---  ------                  --------------   -----
 0   codigo_orgao_superior   438771 non-null  str  
 1   nome_orgao_superior     438771 non-null  str  
 2   codigo_orgao            438771 non-null  str  
 3   nome_orgao              438771 non-null  str  
 4   codigo_unidade_gestora  438771 non-null  str  
 5   nome_unidade_gestora    438771 non-null  str  
 6   ano_extrato             438771 non-null  str  
 7   mes_extrato             438771 non-null  str  
 8   cpf_mascarado           438771 non-null  str  
 9   nome_portador           438771 non-null  str  
 10  documento               438771 non-null  str  
 11  nome_favorecido         438771 non-null  str  
 12  descricao_original      438771 non-null  str  
 13  data_original           438771 non-null  str  
 14  valor_original          438771 non-null  str  
 15  arquivo_ori

codigo_orgao_superior        31
nome_orgao_superior          30
codigo_orgao                198
nome_orgao                  197
codigo_unidade_gestora     1374
nome_unidade_gestora       1281
ano_extrato                   3
mes_extrato                  12
cpf_mascarado              9887
nome_portador              9943
documento                 73665
nome_favorecido           65847
descricao_original            8
data_original              1091
valor_original            69967
Name: valores_distintos, dtype: int64

### Verificando campos vazios e tipos de transação

Como usamos `keep_default_na=False`, os campos vazios aparecem como `''`, e não apenas como valores nulos. Por isso, precisamos verificar as duas situações.

No CPGF, uma data ou identificação ausente pode estar relacionada ao sigilo. Não vamos aplicar `dropna()` indiscriminadamente nem excluir repetições exatas, pois isso poderia alterar os totais.

In [6]:
display((brutos[list(COLUNAS.values())].isna() | brutos[list(COLUNAS.values())].eq('')).sum().rename('campos_vazios'))
display(brutos['descricao_original'].value_counts().rename('transacoes'))
print('Repetições exatas nos campos da fonte:', brutos.duplicated(subset=list(COLUNAS.values())).sum())

codigo_orgao_superior          0
nome_orgao_superior            0
codigo_orgao                   0
nome_orgao                     0
codigo_unidade_gestora         0
nome_unidade_gestora           0
ano_extrato                    0
mes_extrato                    0
cpf_mascarado             110172
nome_portador                  0
documento                      0
nome_favorecido                0
descricao_original             0
data_original             110172
valor_original                 0
Name: campos_vazios, dtype: int64

descricao_original
COMPRA A/V - R$ - APRES                285215
Informações protegidas por sigilo      110172
SAQUE CASH/ATM BB                       37886
COMPRA A/V - INT$ - APRES                3429
SAQUE - INT$ - APRES                     2059
CPP LOJISTA TRF P/FATURA - REAL             6
SAQUE - R$ - APRES                          3
COMP A/V-SOL DISP C/CLI-R$ ANT VENC         1
Name: transacoes, dtype: int64

Repetições exatas nos campos da fonte: 94662


## 4. Transformando os dados

### Tratando os valores monetários

Vamos trabalhar com **centavos inteiros**: `1.234,56` passa a ser `123456`. Isso evita cálculos intermediários com ponto flutuante. Valores negativos são preservados e formatos inválidos interrompem o processamento.

Mantemos uma cópia dos dados originais para comparação.

In [7]:
def converter_centavos(serie):
    texto = serie.str.replace(r'^R\$\s*', '', regex=True).str.strip()
    valido = texto.str.fullmatch(r'-?(?:[0-9]+|[0-9]{1,3}(?:\.[0-9]{3})+),[0-9]{2}', na=False)
    exigir(valido.all(), f'Valores monetários inválidos nos índices: {serie.index[~valido].tolist()[:10]}')
    numeros = texto.str.replace('.', '', regex=False).str.replace(',', '', regex=False).map(int)
    exigir(numeros.abs().le(999999999999999).all(), 'Valor fora da precisão prevista em DECIMAL(15,2).')
    return numeros.astype('int64')

In [8]:
dados = brutos.copy()
obrigatorias = [c for c in COLUNAS.values() if c not in {'data_original', 'cpf_mascarado', 'nome_portador'}]
exigir(dados[obrigatorias].ne('').all().all(), 'Há campos obrigatórios vazios na entrada.')
dados['valor_centavos'] = converter_centavos(dados['valor_original'])

display(dados[['valor_original', 'valor_centavos']].head())

,valor_original,valor_centavos
0,"162,48",16248
1,"28,40",2840
2,"133,10",13310
3,"855,00",85500
4,"35,00",3500


### Identificando registros sigilosos

Usamos os marcadores administrativos exatos e o documento `-11`. Não basta encontrar a palavra “sigilo” em qualquer trecho de um nome.

A identificação de sigilo será usada para atribuir o membro especial `-1` à data e ao portador.

In [9]:
MARCADORES_SIGILO = {'sigiloso', 'informações protegidas por sigilo'}

dados['sigilosa'] = (
    dados['descricao_original'].str.casefold().isin(MARCADORES_SIGILO)
    | dados['nome_portador'].str.casefold().isin(MARCADORES_SIGILO)
    | dados['nome_favorecido'].str.casefold().isin(MARCADORES_SIGILO)
    | dados['documento'].eq('-11')
)

print('Transações sigilosas:', int(dados['sigilosa'].sum()))

Transações sigilosas: 110172


### Convertendo datas e o mês do extrato

A data da transação deve seguir `dd/mm/aaaa`. Datas preenchidas inválidas geram erro; datas ausentes ou sigilosas recebem a chave `-1`.

O mês do extrato é representado pelo primeiro dia do mês. Ele permite analisar o período de prestação de contas mesmo quando a data da transação não está disponível.

In [10]:
datas = pd.to_datetime(dados['data_original'], format='%d/%m/%Y', errors='coerce')
preenchidas = dados['data_original'].ne('')
formato = dados['data_original'].str.fullmatch(r'[0-9]{2}/[0-9]{2}/[0-9]{4}')
invalidas = preenchidas & (datas.isna() | ~formato)
exigir(~invalidas.any(), f'Datas inválidas nos índices: {dados.index[invalidas].tolist()[:10]}')
ano = dados['ano_extrato']
mes = dados['mes_extrato']
exigir(ano.str.fullmatch(r'[0-9]{4}').all() and mes.str.fullmatch(r'[0-9]{1,2}').all(),
       'Ano ou mês do extrato fora do formato esperado.')
exigir(ano.astype(int).isin(ANOS).all() and mes.astype(int).between(1, 12).all(),
       'Ano ou mês do extrato fora do intervalo esperado.')
dados['data_transacao'] = datas.mask(dados['sigilosa'])
dados['data_extrato'] = pd.to_datetime(ano + '-' + mes.str.zfill(2) + '-01', format='%Y-%m-%d')
dados['id_data_transacao'] = dados['data_transacao'].dt.strftime('%Y%m%d').fillna('-1').astype('int64')
dados['id_mes_extrato'] = dados['data_extrato'].dt.strftime('%Y%m%d').astype('int64')

display(dados[['data_original', 'id_data_transacao', 'id_mes_extrato']].head())

,data_original,id_data_transacao,id_mes_extrato
0,29/11/2022,20221129,20230101
1,08/12/2022,20221208,20230101
2,07/12/2022,20221207,20230101
3,05/12/2022,20221205,20230101
4,30/11/2022,20221130,20230101


### Tratando a identificação do portador

Portadores sigilosos ou com CPF e nome vazios apontarão para um membro especial. Se apenas um desses campos estiver ausente fora do sigilo, interrompemos a execução para revisar a origem.

In [11]:
cpf_vazio = dados['cpf_mascarado'].eq('')
nome_vazio = dados['nome_portador'].eq('')
exigir(~((cpf_vazio ^ nome_vazio) & ~dados['sigilosa']).any(),
       'Portador parcialmente identificado fora do sigilo: revisar antes de carregar.')
dados['portador_especial'] = dados['sigilosa'] | (cpf_vazio & nome_vazio)
print('Portadores especiais:', int(dados['portador_especial'].sum()))

Portadores especiais: 110172


### Conferindo o resultado da transformação

Nenhuma linha foi removida. Os valores originais permanecem disponíveis ao lado das colunas tratadas.

In [12]:
exigir(len(dados) == len(brutos), 'A transformação alterou a quantidade de registros.')
display(dados[['ano_extrato', 'mes_extrato', 'valor_original', 'valor_centavos', 'sigilosa']].head())
print('Registros tratados:', len(dados))
print('Total em centavos:', int(dados['valor_centavos'].sum()))

,ano_extrato,mes_extrato,valor_original,valor_centavos,sigilosa
0,2023,01,"162,48",16248,False
1,2023,01,"28,40",2840,False
2,2023,01,"133,10",13310,False
3,2023,01,"855,00",85500,False
4,2023,01,"35,00",3500,False


Registros tratados: 438771
Total em centavos: 32477451409


## 5. Construindo o esquema estrela

Vamos criar cinco dimensões: **órgão, portador, favorecido, tipo de transação e tempo**. As dimensões reúnem descrições; a fato registra cada transação e suas medidas.

Removeremos combinações repetidas apenas nas dimensões. As chaves substitutas (SKs) serão inteiros positivos, exceto os membros especiais `-1`. Como a carga é completa, essas chaves podem mudar se a entrada mudar.

### Dimensão órgão — quem realizou a despesa?

Reunimos unidade gestora, órgão e órgão superior. Os três códigos formam a chave natural. Se a mesma combinação tiver nomes diferentes, precisamos revisar a regra de negócio antes de carregar.

In [13]:
CHAVE_ORGAO = ['codigo_unidade_gestora', 'codigo_orgao', 'codigo_orgao_superior']
CHAVE_PORTADOR = ['cpf_mascarado', 'nome_portador']
CHAVE_FAVORECIDO = ['documento', 'nome_favorecido']

colunas_orgao = CHAVE_ORGAO + ['nome_unidade_gestora', 'nome_orgao', 'nome_orgao_superior']
dim_orgao = dados[colunas_orgao].drop_duplicates()
exigir(~dim_orgao.duplicated(CHAVE_ORGAO).any(), 'Há nomes conflitantes para a mesma chave de órgão.')
dim_orgao = dim_orgao.sort_values(CHAVE_ORGAO).reset_index(drop=True)
dim_orgao.insert(0, 'id_orgao', range(1, len(dim_orgao) + 1))
display(dim_orgao.head())
print('Órgãos:', len(dim_orgao))

,id_orgao,codigo_unidade_gestora,codigo_orgao,codigo_orgao_superior,nome_unidade_gestora,nome_orgao,nome_orgao_superior
0,1,110001,20101,20000,SECRETARIA DE ADMINISTRACAO/PR,Presidência da República,Presidência da República
1,2,110096,63000,63000,SUPERINTENDENCIA REG. DE ADMIN. DA 5ª REGIAO,Advocacia-Geral da União - Unidades com víncul...,Advocacia-Geral da União
2,3,110097,63000,63000,SUPERINTENDENCIA REG. DE ADMIN. DA 4ª REGIAO,Advocacia-Geral da União - Unidades com víncul...,Advocacia-Geral da União
3,4,110099,63000,63000,SUPERINTENDENCIA REG. DE ADMIN. DA 3ª REGIAO,Advocacia-Geral da União - Unidades com víncul...,Advocacia-Geral da União
4,5,110101,60000,20000,GABINETE DA VICE-PRESIDENCIA DA REPUBLICA,Gabinete da Vice-Presidência da República,Presidência da República


Órgãos: 1374


### Dimensão portador — quem utilizou o cartão?

Usamos CPF mascarado e nome em conjunto, conforme o modelo do projeto. O CPF mascarado sozinho não identifica uma pessoa de forma única.

Adicionamos o membro `-1` para portadores sigilosos ou não identificados.

In [14]:
dim_portador = dados.loc[~dados['portador_especial'], CHAVE_PORTADOR].drop_duplicates()
dim_portador = dim_portador.sort_values(CHAVE_PORTADOR).reset_index(drop=True)
dim_portador.insert(0, 'id_portador', range(1, len(dim_portador) + 1))
portador_especial = pd.DataFrame([{
    'id_portador': -1, 'cpf_mascarado': 'SIGILOSO',
    'nome_portador': 'Portador Sigiloso / Não Identificado',
}])
dim_portador = pd.concat([portador_especial, dim_portador], ignore_index=True)
display(dim_portador.head())
print('Portadores:', len(dim_portador))

,id_portador,cpf_mascarado,nome_portador
0,-1,SIGILOSO,Portador Sigiloso / Não Identificado
1,1,***.000.021-**,JOAO BATISTA QUEIROZ
2,2,***.000.191-**,FRANCISCO RODRIGUES DE LIMA
3,3,***.000.251-**,WENDELL PEREIRA DA SILVA
4,4,***.000.266-**,JULIO CESAR LOMBELLO


Portadores: 9955


### Dimensão favorecido — quem recebeu o pagamento?

Usamos documento e nome. A classificação PF/PJ considera o formato do documento, sem comprovar sua validade cadastral. Os códigos administrativos `-1`, `-2` e `-11` são classificados como `Especial`; eles não são as chaves substitutas.

In [15]:
def classificar_documento(documento):
    if documento in {'-1', '-2', '-11'}:
        return 'Especial'
    if re.fullmatch(r'\*{3}\.[0-9]{3}\.[0-9]{3}-\*{2}', documento):
        return 'PF'
    if not re.fullmatch(r'[0-9./-]+', documento):
        raise ValueError('Documento fora dos formatos previstos; revisar a origem.')
    digitos = re.sub(r'[./-]', '', documento)
    if len(digitos) == 14:
        return 'PJ'
    if len(digitos) == 11:
        return 'PF'
    raise ValueError('Documento fora dos comprimentos previstos; revisar a origem.')

dim_favorecido = dados[CHAVE_FAVORECIDO].drop_duplicates()
dim_favorecido = dim_favorecido.sort_values(CHAVE_FAVORECIDO).reset_index(drop=True)
dim_favorecido.insert(0, 'id_favorecido', range(1, len(dim_favorecido) + 1))
dim_favorecido['tipo_pessoa'] = dim_favorecido['documento'].map(classificar_documento)
display(dim_favorecido.head())
display(dim_favorecido['tipo_pessoa'].value_counts())

,id_favorecido,documento,nome_favorecido,tipo_pessoa
0,1,-1,SEM INFORMACAO,Especial
1,2,-11,Sigiloso,Especial
2,3,-2,NAO SE APLICA,Especial
3,4,00000000005150,BANCO DO BRASIL SA,PJ
4,5,00000000170003,BANCO DO BRASIL SA,PJ


tipo_pessoa
PJ          67034
PF           6628
Especial        3
Name: count, dtype: int64

### Dimensão tipo de transação — qual foi a operação?

O dicionário associa cada descrição a uma categoria, moeda e canal. Descrições não mapeadas ficam em `Outros` e devem ser revisadas.

`INT$` segue a classificação `USD` já prevista no projeto, mas isso não comprova a moeda de liquidação. Não estamos convertendo os valores entre moedas.

In [16]:
TIPOS = {
    'COMPRA A/V - R$ - APRES': ('Compra', 'BRL', 'Cartão / Fatura'),
    'COMPRA A/V - INT$ - APRES': ('Compra', 'USD', 'Cartão / Fatura'),
    'SAQUE CASH/ATM BB': ('Saque', 'BRL', 'Espécie / Autoatendimento'),
    'SAQUE - R$ - APRES': ('Saque', 'BRL', 'Espécie / Autoatendimento'),
    'SAQUE - INT$ - APRES': ('Saque', 'USD', 'Espécie / Autoatendimento'),
    'Informações protegidas por sigilo': ('Sigiloso', 'Não Aplicável', 'Sigiloso'),
}



dim_tipo_transacao = dados[['descricao_original']].drop_duplicates()
dim_tipo_transacao = dim_tipo_transacao.sort_values('descricao_original').reset_index(drop=True)
dim_tipo_transacao.insert(0, 'id_tipo_transacao', range(1, len(dim_tipo_transacao) + 1))
classificacao = dim_tipo_transacao['descricao_original'].map(
    lambda descricao: TIPOS.get(descricao, ('Outros', 'Não Aplicável', 'Outros')))
dim_tipo_transacao[['categoria', 'moeda', 'canal']] = pd.DataFrame(
    classificacao.tolist(), index=dim_tipo_transacao.index)
display(dim_tipo_transacao)

,id_tipo_transacao,descricao_original,categoria,moeda,canal
0,1,COMP A/V-SOL DISP C/CLI-R$ ANT VENC,Outros,Não Aplicável,Outros
1,2,COMPRA A/V - INT$ - APRES,Compra,USD,Cartão / Fatura
2,3,COMPRA A/V - R$ - APRES,Compra,BRL,Cartão / Fatura
3,4,CPP LOJISTA TRF P/FATURA - REAL,Outros,Não Aplicável,Outros
4,5,Informações protegidas por sigilo,Sigiloso,Não Aplicável,Sigiloso
5,6,SAQUE - INT$ - APRES,Saque,USD,Espécie / Autoatendimento
6,7,SAQUE - R$ - APRES,Saque,BRL,Espécie / Autoatendimento
7,8,SAQUE CASH/ATM BB,Saque,BRL,Espécie / Autoatendimento


### Dimensão tempo — quando aconteceu?

O calendário cobre anos completos, incluindo datas de transações anteriores a 2023 quando presentes. A chave usa `YYYYMMDD`, e o membro `-1` representa data não informada ou sigilosa.

A mesma dimensão será usada duas vezes na fato: para a data da transação e para o mês do extrato. A função abaixo organiza a criação das colunas do calendário.

In [17]:
def criar_calendario(dados):
    limites = pd.concat([dados['data_transacao'], dados['data_extrato']]).dropna()
    inicio = min(min(ANOS), int(limites.dt.year.min()))
    fim = max(max(ANOS), int(limites.dt.year.max()))
    datas = pd.Series(pd.date_range(f'{inicio}-01-01', f'{fim}-12-31'))
    meses = ['Janeiro', 'Fevereiro', 'Março', 'Abril', 'Maio', 'Junho',
             'Julho', 'Agosto', 'Setembro', 'Outubro', 'Novembro', 'Dezembro']
    dias = ['Segunda-feira', 'Terça-feira', 'Quarta-feira', 'Quinta-feira', 'Sexta-feira', 'Sábado', 'Domingo']
    calendario = pd.DataFrame({
        'id_tempo': datas.dt.strftime('%Y%m%d').astype('int64'),
        'data': datas.dt.strftime('%Y-%m-%d'),
        'dia': datas.dt.day, 'mes': datas.dt.month,
        'nome_mes': datas.dt.month.map(dict(enumerate(meses, 1))),
        'trimestre': datas.dt.quarter, 'semestre': (datas.dt.month - 1) // 6 + 1,
        'ano': datas.dt.year,
        'dia_semana': datas.dt.dayofweek.map(dict(enumerate(dias))),
        'fim_de_semana': datas.dt.dayofweek.ge(5).astype(int),
    })
    especial = {col: None for col in calendario.columns}
    especial.update(id_tempo=-1, nome_mes='Não Informado / Sigiloso', dia_semana='Não Informado')
    calendario = pd.concat([pd.DataFrame([especial]), calendario], ignore_index=True)
    for col in ['id_tempo', 'dia', 'mes', 'trimestre', 'semestre', 'ano', 'fim_de_semana']:
        calendario[col] = calendario[col].astype('Int64')
    return calendario

dim_tempo = criar_calendario(dados)
display(dim_tempo.head())
print('Membros do calendário:', len(dim_tempo))

/tmp/ipykernel_57896/2811118371.py:16: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  'dia_semana': datas.dt.dayofweek.map(dict(enumerate(dias))),
/tmp/ipykernel_57896/2811118371.py:17: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  'fim_de_semana': datas.dt.dayofweek.ge(5).astype(int),


,id_tempo,data,dia,mes,nome_mes,trimestre,semestre,ano,dia_semana,fim_de_semana
0,-1,None,<NA>,<NA>,Não Informado / Sigiloso,<NA>,<NA>,<NA>,Não Informado,<NA>
1,20220101,2022-01-01,1,1,Janeiro,1,1,2022,Sábado,1
2,20220102,2022-01-02,2,1,Janeiro,1,1,2022,Domingo,1
3,20220103,2022-01-03,3,1,Janeiro,1,1,2022,Segunda-feira,0
4,20220104,2022-01-04,4,1,Janeiro,1,1,2022,Terça-feira,0


Membros do calendário: 1462


### Reunindo as dimensões para a carga

Os nomes das tabelas e colunas seguem o contrato em `scripts/sql/ddl/schema.sql`.

In [18]:
dimensoes = {
    'DIM_TEMPO': dim_tempo,
    'DIM_ORGAO': dim_orgao,
    'DIM_PORTADOR': dim_portador,
    'DIM_FAVORECIDO': dim_favorecido,
    'DIM_TIPO_TRANSACAO': dim_tipo_transacao,
}
display(pd.Series({nome: len(dim) for nome, dim in dimensoes.items()}, name='membros'))

DIM_TEMPO              1462
DIM_ORGAO              1374
DIM_PORTADOR           9955
DIM_FAVORECIDO        73665
DIM_TIPO_TRANSACAO        8
Name: membros, dtype: int64

## 6. Montando a tabela fato

Vamos associar cada transação às chaves das dimensões com `merge`. A opção `many_to_one` verifica que cada registro encontra no máximo um membro na dimensão, evitando multiplicar despesas.

Primeiro, ajustamos a identificação dos portadores especiais apenas na cópia usada nas junções.

In [19]:
base = dados.copy()
base['_ordem'] = range(len(base))
base.loc[base['portador_especial'], 'cpf_mascarado'] = 'SIGILOSO'
base.loc[base['portador_especial'], 'nome_portador'] = 'Portador Sigiloso / Não Identificado'

base = base.merge(dim_orgao[CHAVE_ORGAO + ['id_orgao']], on=CHAVE_ORGAO, how='left', validate='many_to_one')
base = base.merge(dim_portador[CHAVE_PORTADOR + ['id_portador']], on=CHAVE_PORTADOR, how='left', validate='many_to_one')
base = base.merge(dim_favorecido[CHAVE_FAVORECIDO + ['id_favorecido']], on=CHAVE_FAVORECIDO, how='left', validate='many_to_one')
base = base.merge(dim_tipo_transacao, on='descricao_original', how='left', validate='many_to_one')
base = base.sort_values('_ordem').reset_index(drop=True)

chaves = ['id_orgao', 'id_portador', 'id_favorecido', 'id_tipo_transacao']
exigir(len(base) == len(dados), 'As junções alteraram a quantidade de registros.')
exigir(base[chaves].notna().all().all(), 'Há transações sem correspondência nas dimensões.')
base[chaves] = base[chaves].astype('int64')
display(base[chaves].head())

,id_orgao,id_portador,id_favorecido,id_tipo_transacao
0,8,5642,40946,3
1,8,8672,11103,3
2,8,8672,11103,3
3,8,9840,6584,3
4,8,3379,16226,3


### Selecionando as chaves e medidas

Cada linha tem `quantidade_transacoes = 1`. O identificador da transação será gerado pelo SQLite.

Para respeitar o esquema existente, enviamos o valor como texto decimal para a coluna `DECIMAL(15,2)`. O SQLite não garante armazenamento decimal exato: por isso conferiremos os valores em centavos após a carga. Nas consultas financeiras, arredondamos cada valor para centavos antes de somar.

In [20]:
COLUNAS_FATO = ['id_data_transacao', 'id_mes_extrato', 'id_orgao', 'id_portador',
                'id_favorecido', 'id_tipo_transacao', 'valor_transacao', 'quantidade_transacoes']
base['valor_transacao'] = base['valor_centavos'].map(lambda n: format(Decimal(int(n)) / 100, '.2f'))
base['quantidade_transacoes'] = 1
fato = base[COLUNAS_FATO].copy()
auditoria = base

exigir(base['valor_centavos'].equals(dados['valor_centavos'].reset_index(drop=True)),
       'Os valores ou a ordem mudaram nas junções.')
for coluna in ['id_data_transacao', 'id_mes_extrato']:
    exigir(fato[coluna].isin(dim_tempo['id_tempo']).all(), 'Calendário incompleto.')
exigir(fato.loc[dados['portador_especial'].to_numpy(), 'id_portador'].eq(-1).all(),
       'Portador especial sem a chave -1.')
display(fato.head())
print('Registros na fato:', len(fato))

,id_data_transacao,id_mes_extrato,id_orgao,id_portador,id_favorecido,id_tipo_transacao,valor_transacao,quantidade_transacoes
0,20221129,20230101,8,5642,40946,3,162.48,1
1,20221208,20230101,8,8672,11103,3,28.40,1
2,20221207,20230101,8,8672,11103,3,133.10,1
3,20221205,20230101,8,9840,6584,3,855.00,1
4,20221130,20230101,8,3379,16226,3,35.00,1


Registros na fato: 438771


## 7. Carregando o Data Warehouse no SQLite

A carga é completa: reconstruímos dimensões e fato juntas. As funções auxiliares abaixo cuidam da inserção e da conferência, sem realizar novas transformações de negócio.

Criamos as tabelas com o SQL do projeto para preservar chaves e restrições. Carregamos dimensões antes da fato, recriamos os índices após a inserção e verificamos contagens, integridade, relacionamentos e totais por mês/órgão.

O banco é criado em um arquivo temporário. Só depois de validado ele substitui o destino. Feche conexões externas antes de executar a carga. No modo completo, o destino é `data/database/cpgf_dw.sqlite`; no modo amostra, `cpgf_dw_amostra.sqlite`.

In [21]:
def inserir_dataframe(conexao, tabela, df):
    # Nomes vêm exclusivamente dos DataFrames e tabelas definidos neste notebook.
    colunas = ', '.join(f'"{c}"' for c in df.columns)
    parametros = ', '.join('?' for _ in df.columns)
    sql = f'INSERT INTO "{tabela}" ({colunas}) VALUES ({parametros})'
    registros = df.astype(object).where(df.notna(), None).itertuples(index=False, name=None)
    conexao.executemany(sql, registros)

def validar_banco(conexao, dimensoes, fato, auditoria):
    for tabela, df in {**dimensoes, 'FATO_TRANSACAO_CPGF': fato}.items():
        quantidade = conexao.execute(f'SELECT COUNT(*) FROM "{tabela}"').fetchone()[0]
        exigir(quantidade == len(df), f'{tabela}: contagem após carga divergente.')
    exigir(not conexao.execute('PRAGMA foreign_key_check').fetchall(), 'Há chaves estrangeiras inválidas.')
    exigir(conexao.execute('PRAGMA integrity_check').fetchall() == [('ok',)], 'Falha na integridade SQLite.')
    # Confere o valor efetivamente armazenado de cada linha, em ordem de inserção.
    centavos_db = conexao.execute(
        'SELECT CAST(ROUND(valor_transacao * 100) AS INTEGER) FROM FATO_TRANSACAO_CPGF ORDER BY id_transacao')
    exigir(all(linha[0] == int(valor) for linha, valor in zip(centavos_db, auditoria['valor_centavos'])),
           'Perda de precisão monetária no armazenamento.')
    obtido = pd.read_sql_query("""
        SELECT id_mes_extrato, id_orgao, COUNT(*) AS transacoes,
               SUM(CAST(ROUND(valor_transacao * 100) AS INTEGER)) AS centavos
        FROM FATO_TRANSACAO_CPGF
        GROUP BY id_mes_extrato, id_orgao
        ORDER BY id_mes_extrato, id_orgao
    """, conexao)
    esperado = auditoria.groupby(['id_mes_extrato', 'id_orgao'], as_index=False).agg(
        transacoes=('valor_centavos', 'size'), centavos=('valor_centavos', 'sum'))
    pd.testing.assert_frame_equal(obtido, esperado, check_dtype=False)
    return obtido

def carregar_dw(destino, ddl, dimensoes, fato, auditoria):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    # Um banco aberto em WAL pode ter estado fora do arquivo principal.
    exigir(not any(Path(str(destino) + sufixo).exists() for sufixo in ('-wal', '-shm', '-journal')),
           'Há arquivos auxiliares do banco destino. Feche as conexões e revise antes da substituição.')
    with tempfile.NamedTemporaryFile(dir=destino.parent, prefix='.cpgf_etl_', suffix='.sqlite', delete=False) as arq:
        temporario = Path(arq.name)
    try:
        conexao = sqlite3.connect(temporario)
        try:
            conexao.execute('PRAGMA foreign_keys = ON')
            conexao.executescript(Path(ddl).read_text(encoding='utf-8'))
            indices = conexao.execute("SELECT name, sql FROM sqlite_master WHERE type='index' AND sql IS NOT NULL").fetchall()
            for nome, _ in indices:
                conexao.execute(f'DROP INDEX "{nome}"')
            with conexao:
                for tabela, df in dimensoes.items():
                    inserir_dataframe(conexao, tabela, df)
                inserir_dataframe(conexao, 'FATO_TRANSACAO_CPGF', fato)
                for _, sql in indices:
                    conexao.execute(sql)
                resumo = validar_banco(conexao, dimensoes, fato, auditoria)
        finally:
            conexao.close()
        temporario.replace(destino)
    finally:
        temporario.unlink(missing_ok=True)
    return resumo

### Executando a carga

Se alguma validação falhar, a execução é interrompida e o banco anterior permanece preservado.

In [22]:
inicio_carga = time.perf_counter()
resumo_carregado = carregar_dw(DESTINO, DDL, dimensoes, fato, auditoria)
print('DW carregado e validado:', DESTINO)
print('Tempo da carga e validação (s):', round(time.perf_counter() - inicio_carga, 2))

DW carregado e validado: /home/fagpaf/Projetos/projeto-integracao-dw/data/database/cpgf_dw.sqlite
Tempo da carga e validação (s): 4.76


## 8. Conferindo o resultado final

Vamos comparar a quantidade e o total financeiro de entrada com o que foi consultado no banco. Esses números precisam coincidir.

In [23]:
comparacao = pd.DataFrame({
    'etapa': ['Antes da carga', 'Depois da carga'],
    'transacoes': [len(dados), int(resumo_carregado['transacoes'].sum())],
    'total_centavos': [int(dados['valor_centavos'].sum()), int(resumo_carregado['centavos'].sum())],
})
display(comparacao)
display(resumo_carregado.head())

,etapa,transacoes,total_centavos
0,Antes da carga,438771,32477451409
1,Depois da carga,438771,32477451409


,id_mes_extrato,id_orgao,transacoes,centavos
0,20230101,1,266,26659379
1,20230101,2,15,861168
2,20230101,3,4,243830
3,20230101,4,6,273500
4,20230101,5,94,11032832


### Exemplo de análise: valores por órgão e ano do extrato

Esta consulta mostra como a fato se relaciona às dimensões. O total representa a soma dos valores fornecidos pela origem, sem conversão cambial.

In [24]:
with sqlite3.connect(f'{DESTINO.resolve().as_uri()}?mode=ro', uri=True) as conexao:
    gastos_por_orgao = pd.read_sql_query("""
        SELECT t.ano, o.codigo_orgao, o.nome_orgao,
               COUNT(*) AS quantidade_transacoes,
               SUM(CAST(ROUND(f.valor_transacao * 100) AS INTEGER)) AS total_centavos
        FROM FATO_TRANSACAO_CPGF AS f
        JOIN DIM_ORGAO AS o ON f.id_orgao = o.id_orgao
        JOIN DIM_TEMPO AS t ON f.id_mes_extrato = t.id_tempo
        GROUP BY t.ano, o.codigo_orgao, o.nome_orgao
        ORDER BY total_centavos DESC
        LIMIT 10
    """, conexao)
display(gastos_por_orgao)

,ano,codigo_orgao,nome_orgao,quantidade_transacoes,total_centavos
0,2025,30108,Polícia Federal,31819,3344187634
1,2024,30108,Polícia Federal,32252,3286072285
2,2024,20101,Presidência da República,10823,2603850833
3,2023,30108,Polícia Federal,25281,2418022313
4,2023,20101,Presidência da República,11130,2368144845
5,2025,20101,Presidência da República,11541,2257477376
6,2024,52121,Comando do Exército,14223,1140750940
7,2025,52121,Comando do Exército,15480,1078209807
8,2023,25205,Fundação Instituto Brasileiro de Geografia e E...,21428,884296103
9,2025,25205,Fundação Instituto Brasileiro de Geografia e E...,19374,884026916
